# Lab: Can an AI weather model forecast your event?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/teaching/blob/main/ai_weather_lab/neuralgcm_event_lab.ipynb)

Pick a weather event you care about. You will look at it in ERA5, see how much of it survives on a coarse model grid, and find out whether **NeuralGCM** can forecast it, and how many days ahead.

NeuralGCM (Kochkov et al. 2024, *Nature*) is a hybrid model: a conventional dynamical core plus neural networks trained on ERA5 for everything else, such as clouds, convection and rain.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the four setup cells. Everything you change is plain code: type your values straight into the cells.

In [ ]:
#@title Setup: install packages (~1 min)
!pip install -q neuralgcm gcsfs cartopy
# If the next cell fails with a version error, uncomment this line, run it, then
# restart the session (Runtime -> Restart session) and continue from the next cell:
# !pip install -q "neuralgcm==1.2.2" "dinosaur==1.3.6"

In [ ]:
#@title Setup: imports and ERA5 data
import pickle
import time

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import gcsfs
import jax
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from dinosaur import horizontal_interpolation, spherical_harmonic, xarray_utils
from IPython.display import HTML, display
from matplotlib import animation, colors

import neuralgcm

print("JAX devices:", jax.devices())
if jax.default_backend() != "gpu":
    print("\n*** No GPU found. Use Runtime -> Change runtime type -> GPU, then rerun. ***")

plt.rcParams.update({"font.size": 11, "figure.dpi": 100, "animation.embed_limit": 100})
G = 9.80665   # gravity, m/s^2
RD = 287.05   # gas constant of dry air, J/(kg K)

# ERA5 reanalysis, hourly at 0.25 degrees (Google's public copy, no account needed)
ERA5 = xr.open_zarr("gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3",
                    chunks=None, storage_options=dict(token="anon"))


def crop(ds, lats, lons):
    # Cut out a box: lats = (south, north), lons = (west, east), with negative longitudes to the west.
    # Longitudes in the result run continuously from west to east, even across 0 or 180 degrees.
    west, east = lons
    offset = (ds.longitude - west) % 360
    ds = ds.isel(longitude=np.flatnonzero(offset.values <= (east - west) % 360))
    ds = ds.assign_coords(longitude=west + (ds.longitude - west) % 360).sortby("longitude")
    inside = (ds.latitude.values >= lats[0]) & (ds.latitude.values <= lats[1])
    return ds.isel(latitude=np.flatnonzero(inside)).sortby("latitude")


def add_wind_speed(ds):
    return ds.assign(wind=np.hypot(ds.u, ds.v))


def era5_fields(start, end, every_hours=6):
    # ERA5 weather in the box (LATS, LONS), every `every_hours` hours from start to end
    times = np.arange(np.datetime64(start), np.datetime64(end) + np.timedelta64(1, "h"),
                      np.timedelta64(every_hours, "h"))
    names = ["total_column_cloud_liquid_water", "total_column_cloud_ice_water", "total_column_water_vapour",
             "total_precipitation", "mean_sea_level_pressure", "2m_temperature",
             "10m_u_component_of_wind", "10m_v_component_of_wind"]
    tic = time.time()
    raw = crop(ERA5[names].sel(time=times), LATS, LONS).compute()
    print(f"Loaded {len(times)} times of ERA5 in {time.time() - tic:.0f} s")
    return add_wind_speed(xr.Dataset({
        "clouds": raw.total_column_cloud_liquid_water + raw.total_column_cloud_ice_water,  # kg/m^2
        "moisture": raw.total_column_water_vapour,       # kg/m^2, the same as mm of water
        "rain": raw.total_precipitation * 1000,          # mm in the hour before each time, so mm/h
        "pressure": raw.mean_sea_level_pressure / 100,   # hPa
        "temperature": raw["2m_temperature"] - 273.15,   # deg C, 2 m above the ground
        "u": raw["10m_u_component_of_wind"],             # m/s, 10 m above the ground
        "v": raw["10m_v_component_of_wind"],
    }))

In [ ]:
#@title Setup: maps and animations
VIEWS = {
    # name: (color bar label, color map, color scale; None fits the scale to the data)
    "clouds": ("cloud water and ice in the column (kg/m²)", "gray", colors.PowerNorm(0.5, vmin=0, vmax=1.5)),
    "moisture": ("water vapor in the column (mm)", "YlGnBu", colors.Normalize(0, 70)),
    "rain": ("rain (mm/h)", "turbo", colors.LogNorm(0.1, 30)),
    "temperature": ("temperature near the ground (°C)", "RdYlBu_r", None),
    "wind": ("wind speed near the ground (m/s)", "magma_r", colors.Normalize(0, 40)),
}
TICKS = {"clouds": [0, 0.1, 0.25, 0.5, 1, 1.5], "rain": [0.1, 0.3, 1, 3, 10, 30]}
PC = ccrs.PlateCarree()


def when(t):
    return pd.Timestamp(t).strftime("%b %d %Y, %H UTC")


def box_projection():
    # Map projection centered on the box, and the box's extent in it
    width = (LONS[1] - LONS[0]) % 360
    center = LONS[0] + width / 2
    return ccrs.PlateCarree(central_longitude=center), [-width / 2, width / 2, LATS[0], LATS[1]]


def map_figure(n, ncols=None, panel_width=5.5):
    # A figure with n maps of the box
    ncols = ncols or n
    nrows = int(np.ceil(n / ncols))
    proj, extent = box_projection()
    panel_height = panel_width * (extent[3] - extent[2]) / (extent[1] - extent[0])
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_width * ncols, panel_height * nrows + 1.5),
                             subplot_kw={"projection": proj}, layout="constrained", squeeze=False)
    for ax in axes.flat[n:]:
        ax.remove()
    return fig, list(axes.flat[:n])


def lowest_pressure(pressure):
    # The lowest sea-level pressure on a map, with its latitude and longitude
    p = pressure.stack(point=("latitude", "longitude"))
    return p.isel(point=p.argmin("point"))


def low_center(p, lat, lon, radius_deg):
    # Center of a low near (lat, lon): the average position of the points within radius_deg,
    # weighted by how much lower their pressure is than the average there. Smoother than the lowest grid point.
    near = (np.abs(p.latitude - lat) <= radius_deg) & (np.abs(p.longitude - lon) <= radius_deg)
    depth = (p.where(near).mean() - p).where(near).clip(min=0) ** 2
    return float((depth * p.latitude).sum() / depth.sum()), float((depth * p.longitude).sum() / depth.sum())


def storm_track(ds, lat, lon, search_deg=7):
    # Follow a low from (lat, lon): at each time, find the lowest pressure within search_deg of its last
    # position, then the center of the low around it
    track = []
    for t in ds.time.values:
        p = ds.pressure.sel(time=t)
        near = (np.abs(p.latitude - lat) <= search_deg) & (np.abs(p.longitude - lon) <= search_deg)
        low = lowest_pressure(p.where(near))
        lat, lon = low_center(p, float(low.latitude), float(low.longitude), radius_deg=4)
        track.append((lat, lon))
    return np.array(track)


def add_colorbar(fig, axes, view, norm, **kwargs):
    fig.colorbar(plt.cm.ScalarMappable(norm, VIEWS[view][1]), ax=axes, orientation="horizontal",
                 label=VIEWS[view][0], ticks=TICKS.get(view), format="%g", **kwargs)


def format_dates(ax):
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))


def color_scale(datasets, view):
    if VIEWS[view][2] is not None:
        return VIEWS[view][2]
    values = [ds[view] for ds in datasets if view in ds]
    return colors.Normalize(min(float(v.min()) for v in values), max(float(v.max()) for v in values))


def draw_map(ax, ds, view, norm):
    # One map at one time: the view in color, sea-level pressure contours every 4 hPa, and the lowest pressure
    label, cmap, _ = VIEWS[view]
    proj, extent = box_projection()
    dark = view == "clouds"   # clouds look like a satellite image: white on black
    if not dark:
        ax.add_feature(cfeature.OCEAN, color="#eef3f7")
        ax.add_feature(cfeature.LAND, color="#e3e3e3")
    if view in ds:
        field = ds[view].where(ds[view] >= 0.1) if view == "rain" else ds[view]   # leave dry areas blank
        ax.pcolormesh(field.longitude, field.latitude, field.transpose("latitude", "longitude"),
                      transform=PC, cmap=cmap, norm=norm, shading="auto")
    else:
        ax.text(0.5, 0.5, f"this model has no {view}", transform=ax.transAxes, ha="center", fontsize=12)
    if view == "wind":
        step = max(1, ds.sizes["longitude"] // 15)   # about 15 arrows across
        w = ds.isel(latitude=slice(None, None, step), longitude=slice(None, None, step))
        ax.quiver(w.longitude.values, w.latitude.values, w.u.transpose("latitude", "longitude").values,
                  w.v.transpose("latitude", "longitude").values, transform=PC, color="0.2", width=0.003,
                  scale=400, scale_units="width")   # the same arrow length for the same speed on every map
    lines = "white" if dark else "black"
    p = ds.pressure.transpose("latitude", "longitude")
    ax.contour(p.longitude, p.latitude, p, levels=np.arange(900, 1080, 4), colors=lines,
               linewidths=0.6, transform=PC)
    low = lowest_pressure(ds.pressure)
    ax.text(float(low.longitude), float(low.latitude), f"L {float(low):.0f}", transform=PC, color=lines,
            ha="center", va="center", fontsize=11, fontweight="bold",
            bbox=dict(boxstyle="round", fc="black" if dark else "white", ec="none", alpha=0.6))
    ax.coastlines(color="gold" if dark else "0.2", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, edgecolor="gold" if dark else "0.3", linewidth=0.5)
    ax.add_feature(cfeature.STATES, edgecolor="gold" if dark else "0.5", linewidth=0.3)
    ax.set_extent(extent, crs=proj)


def show_views(ds, time):
    # All views of one dataset at one time
    ds = crop(ds, LATS, LONS).sel(time=np.datetime64(time), method="nearest")
    fig, axes = map_figure(len(VIEWS), ncols=3)
    for ax, view in zip(axes, VIEWS):
        norm = color_scale([ds], view)
        draw_map(ax, ds, view, norm)
        add_colorbar(fig, ax, view, norm, shrink=0.8)
    fig.suptitle(f"{EVENT}, {when(ds.time.values)}")
    plt.show()


def animate(runs, view):
    # Animated maps of one view, one panel per dataset in `runs` ({name: dataset}), at the times they share
    runs = {name: crop(ds, LATS, LONS) for name, ds in runs.items()}
    times = sorted(set.intersection(*(set(ds.time.values) for ds in runs.values())))
    if not times:
        print("These datasets share no times. Check START, END and the forecast start.")
        return
    norm = color_scale(runs.values(), view)
    fig, axes = map_figure(len(runs))
    add_colorbar(fig, axes, view, norm, shrink=0.6)

    def frame(i):
        for ax, (name, ds) in zip(axes, runs.items()):
            ax.clear()
            draw_map(ax, ds.sel(time=times[i]), view, norm)
            ax.set_title(name)
        fig.suptitle(f"{EVENT}, {when(times[i])}")

    print(f"Building the animation ({len(times)} frames)...")
    frame(0)
    anim = animation.FuncAnimation(fig, frame, frames=len(times), interval=400)
    display(HTML(anim.to_jshtml(default_mode="once")))
    plt.close(fig)


def show_resolutions(ds, time, view):
    # One time at ERA5's 0.25 degrees, and averaged onto coarser grids like NeuralGCM's
    ds = crop(ds, LATS, LONS).sel(time=np.datetime64(time), method="nearest")
    grids = {"ERA5, 0.25°": ds}
    for k, model_grid in [(3, "0.7°"), (6, "1.4°"), (11, "2.8°")]:
        coarse = ds.drop_vars("wind").coarsen(latitude=k, longitude=k, boundary="trim").mean()
        grids[f"{0.25 * k:g}° (like NeuralGCM {model_grid})"] = add_wind_speed(coarse)
    norm = color_scale([ds], view)
    fig, axes = map_figure(len(grids), ncols=2)
    for ax, (name, g) in zip(axes, grids.items()):
        draw_map(ax, g, view, norm)
        ax.set_title(name)
    add_colorbar(fig, axes, view, norm, shrink=0.6)
    fig.suptitle(f"{EVENT}, {when(ds.time.values)}")
    plt.show()
    print(f"{'grid':28s} {'lowest pressure':>16s} {'strongest wind':>15s} {'heaviest rain':>14s}")
    for name, g in grids.items():
        print(f"{name:28s} {float(g.pressure.min()):12.0f} hPa {float(g.wind.max()):11.0f} m/s "
              f"{float(g.rain.max()):9.1f} mm/h")


def run_colors(runs):
    return {name: "black" if name == "ERA5" else f"C{i}" for i, name in enumerate(runs)}


def plot_tracks(runs):
    # Track of the low that is deepest in ERA5 at START, in ERA5 and in each forecast, every 6 hours
    start = np.datetime64(START)
    first = lowest_pressure(crop(runs["ERA5"], LATS, LONS).pressure.sel(time=start))
    fig, axes = map_figure(1, panel_width=8)
    ax = axes[0]
    ax.add_feature(cfeature.OCEAN, color="#eef3f7")
    ax.add_feature(cfeature.LAND, color="#e3e3e3")
    ax.coastlines(color="0.3", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, edgecolor="0.4", linewidth=0.5)
    for (name, ds), color in zip(runs.items(), run_colors(runs).values()):
        track = storm_track(crop(ds, LATS, LONS).sel(time=slice(start, None)),
                            float(first.latitude), float(first.longitude))
        ax.plot(track[:, 1], track[:, 0], "o-", color=color, ms=3, lw=2.5 if name == "ERA5" else 1.5,
                transform=PC, label=name)
    proj, extent = box_projection()
    ax.set_extent(extent, crs=proj)
    ax.legend(loc="best", fontsize=9)
    ax.set_title(f"{EVENT}: track of the lowest pressure")
    plt.show()


def plot_box_summary(runs):
    # How strong the event is inside the box over time: ERA5 in black, forecasts in color
    stats = [("lowest pressure (hPa)", "pressure", "min"), ("strongest wind (m/s)", "wind", "max"),
             ("heaviest rain (mm/h)", "rain", "max"), ("average temperature (°C)", "temperature", "mean")]
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.8), layout="constrained")
    for ax, (title, field, how) in zip(axes, stats):
        for (name, ds), color in zip(runs.items(), run_colors(runs).values()):
            ds = crop(ds, LATS, LONS)
            if field in ds:
                series = getattr(ds[field], how)(("latitude", "longitude"))
                ax.plot(series.time, series, color=color, lw=2.5 if name == "ERA5" else 1.5, label=name)
        ax.set_title(title)
        format_dates(ax)
        ax.grid(alpha=0.3)
    axes[0].legend(fontsize=9)
    fig.suptitle(f"{EVENT}: inside the box")
    plt.show()

In [ ]:
#@title Setup: running NeuralGCM
OUTPUT_HOURS = 6   # forecasts are saved every 6 hours
gcs = gcsfs.GCSFileSystem(token="anon")


def load_model(name):
    # Load a NeuralGCM model, and get ready to put ERA5 on its grid
    global model, regridder, surface_geopotential
    with gcs.open(f"gs://neuralgcm/models/{name}", "rb") as f:
        model = neuralgcm.PressureLevelModel.from_checkpoint(pickle.load(f))
    era5_grid = spherical_harmonic.Grid(
        latitude_nodes=ERA5.sizes["latitude"],
        longitude_nodes=ERA5.sizes["longitude"],
        latitude_spacing=xarray_utils.infer_latitude_spacing(ERA5.latitude),
        longitude_offset=xarray_utils.infer_longitude_offset(ERA5.longitude),
    )
    regridder = horizontal_interpolation.ConservativeRegridder(
        era5_grid, model.data_coords.horizontal, skipna=True)
    surface = to_model_grid(ERA5[["geopotential_at_surface"]].sel(time=["2000-01-01T00"]).compute())
    surface_geopotential = surface.geopotential_at_surface.isel(time=0, drop=True)
    grid = model.data_coords.horizontal
    print(f"Loaded {name} ({grid.longitude_nodes} x {grid.latitude_nodes} grid points)")


def to_model_grid(ds):
    # Average ERA5 onto the model's grid
    return xarray_utils.fill_nan_with_nearest(xarray_utils.regrid(ds, regridder))


def load_initial_state(start):
    # The ERA5 state at `start`, on the model grid
    raw = (
        ERA5[model.input_variables + model.forcing_variables]
        # The model expects sea surface temperature and sea ice from 24 h earlier, as in training
        .pipe(xarray_utils.selective_temporal_shift,
              variables=model.forcing_variables, time_shift="24 hours")
        .sel(time=[np.datetime64(start)])
        .compute()
    )
    return to_model_grid(raw)


def add_warm_blob(state, lat, lon, kelvin, radius_km=500):
    # Warm the air between 850 and 300 hPa by `kelvin` at (lat, lon), fading out over radius_km
    lat_r, lon_r = np.deg2rad(state.latitude), np.deg2rad(state.longitude)
    cos_distance = (np.sin(lat_r) * np.sin(np.deg2rad(lat))
                    + np.cos(lat_r) * np.cos(np.deg2rad(lat)) * np.cos(lon_r - np.deg2rad(lon)))
    distance_km = 6371 * np.arccos(np.clip(cos_distance, -1, 1))
    blob = kelvin * np.exp(-(distance_km / radius_km) ** 2)
    in_layer = (state.level >= 300) & (state.level <= 850)
    warm = state.copy()
    warm["temperature"] = (state.temperature + blob.where(in_layer, 0.0)).transpose(*state.temperature.dims)
    return warm


def model_fields(out):
    # The same weather fields as era5_fields, computed from the model's pressure levels
    # Same grid; copying the coordinates avoids tiny rounding differences between them
    surface = surface_geopotential.assign_coords(latitude=out.latitude, longitude=out.longitude)
    above_ground = out.geopotential > surface

    def column(x):   # amount in the column above the ground, kg/m^2
        return x.where(above_ground, 0.0).integrate("level") * 100 / G

    def near_ground(x):   # value on the lowest pressure level that is above the ground
        return x.where(above_ground).sortby("level", ascending=False).bfill("level").isel(level=0)

    z1000, t1000 = out.geopotential.sel(level=1000), out.temperature.sel(level=1000)
    fields = xr.Dataset({
        "clouds": column(out.specific_cloud_liquid_water_content + out.specific_cloud_ice_water_content),
        "moisture": column(out.specific_humidity),
        "pressure": 1000 * np.exp(z1000 / (RD * t1000)),   # hPa, sea-level pressure from the 1000 hPa height
        "temperature": near_ground(out.temperature) - 273.15,
        "u": near_ground(out.u_component_of_wind),
        "v": near_ground(out.v_component_of_wind),
    })
    if "precipitation_cumulative_mean" in out:   # only the v1_precip models predict rain
        total = out.precipitation_cumulative_mean.squeeze("surface", drop=True) * 1000   # mm since the start
        fields["rain"] = (total.diff("time") / OUTPUT_HOURS).reindex(time=out.time)       # mm/h
    return add_wind_speed(fields)


def forecast(start, end, nudge=None, seed=0):
    # Run NeuralGCM from the ERA5 state at `start` until `end`. Returns weather fields for the whole globe.
    # nudge = (lat, lon, kelvin) adds a warm blob to the starting state first.
    tic = time.time()
    state = load_initial_state(start)
    if nudge is not None:
        state = add_warm_blob(state, *nudge)
    print(f"Loaded the ERA5 starting state in {time.time() - tic:.0f} s; running...")
    inputs = model.inputs_from_xarray(state.isel(time=0))
    forcings = model.forcings_from_xarray(state.isel(time=0))
    model_state = model.encode(inputs, forcings, jax.random.key(seed))
    days = int(np.ceil((np.datetime64(end) - np.datetime64(start)) / np.timedelta64(1, "D")))
    pieces = []
    for day in range(days + 1):   # one day at a time, so JAX compiles the model only once
        model_state, out = model.unroll(
            model_state,
            model.forcings_from_xarray(state.head(time=1)),   # sea surface temperature and sea ice stay fixed
            steps=24 // OUTPUT_HOURS,
            timedelta=np.timedelta64(OUTPUT_HOURS, "h"),
            start_with_input=True,
        )
        pieces.append(jax.device_get(out))
    out = jax.tree.map(lambda *days: np.concatenate(days), *pieces)
    hours = np.arange(len(out["temperature"])) * OUTPUT_HOURS
    out = model.data_to_xarray(out, times=hours)
    out = out.assign_coords(time=np.datetime64(start) + hours.astype("timedelta64[h]"))
    fields = model_fields(out.sel(time=slice(None, np.datetime64(end))))
    print(f"Forecast from {when(start)} done in {time.time() - tic:.0f} s")
    return fields


def show_spread(nudged, control, nudge):
    # Where the nudged forecast differs from the control: an animated world map, then the largest
    # difference inside the box over time
    diff = (nudged.pressure - control.pressure).transpose("time", "latitude", "longitude")
    lat0, lon0, kelvin = nudge
    norm = colors.SymLogNorm(linthresh=0.1, vmin=-10, vmax=10)
    fig, ax = plt.subplots(figsize=(10, 5.5), subplot_kw={"projection": ccrs.Robinson(np.mean(LONS))},
                           layout="constrained")
    fig.colorbar(plt.cm.ScalarMappable(norm, "RdBu_r"), ax=ax, orientation="horizontal", shrink=0.6,
                 label="nudged - control sea-level pressure (hPa)", ticks=[-10, -1, -0.1, 0, 0.1, 1, 10],
                 format="%g")

    def frame(i):
        ax.clear()
        d = diff.isel(time=i)
        ax.pcolormesh(d.longitude, d.latitude, d, transform=PC, cmap="RdBu_r", norm=norm, shading="auto")
        ax.coastlines(color="0.3", linewidth=0.6)
        west, east = LONS[0], LONS[0] + (LONS[1] - LONS[0]) % 360
        ax.plot([west, east, east, west, west], [LATS[0], LATS[0], LATS[1], LATS[1], LATS[0]],
                color="black", lw=2, transform=PC)
        ax.plot(lon0, lat0, "*", color="gold", ms=18, mec="black", transform=PC)
        ax.set_global()
        ax.set_title(f"{when(d.time.values)}  (star: {kelvin:g} K warm blob; box: your event)")

    print(f"Building the animation ({diff.sizes['time']} frames)...")
    frame(0)
    anim = animation.FuncAnimation(fig, frame, frames=diff.sizes["time"], interval=400)
    display(HTML(anim.to_jshtml(default_mode="once")))
    plt.close(fig)

    in_box = np.abs(crop(diff, LATS, LONS)).max(("latitude", "longitude"))
    fig, ax = plt.subplots(figsize=(7, 3.8), layout="constrained")
    ax.semilogy(in_box.time, in_box.where(in_box > 0), "o-", lw=2)
    ax.set(title="Largest pressure difference inside your box", ylabel="hPa (log scale)")
    format_dates(ax)
    ax.grid(alpha=0.3, which="both")
    plt.show()

---
## 1. Choose an event

Pick a time window and a box around your event. Times are UTC; start and end on 00, 06, 12 or 18 h. Longitudes west of Greenwich are negative. ERA5 goes back to 1940.

In [ ]:
EVENT = "Hurricane Sandy"
START = "2012-10-25T00"   # first time to look at
END = "2012-10-31T00"     # last time
LATS = (20, 50)           # south, north
LONS = (-85, -55)         # west, east

# Other events to try:
# Typhoon Tip:                    START = "1979-10-08T00", END = "1979-10-16T00", LATS = (5, 40),  LONS = (120, 160)
# 1993 Storm of the Century:      START = "1993-03-11T00", END = "1993-03-15T00", LATS = (20, 55), LONS = (-100, -60)
# Texas cold-air outbreak:        START = "2021-02-10T00", END = "2021-02-18T00", LATS = (20, 55), LONS = (-115, -80)
# Pacific Northwest heat dome:    START = "2021-06-24T00", END = "2021-07-02T00", LATS = (35, 60), LONS = (-135, -105)
# California atmospheric rivers:  START = "2023-01-01T00", END = "2023-01-11T00", LATS = (25, 50), LONS = (-150, -110)

---
## 2. Look at it in ERA5

ERA5 is a reanalysis: a model run constrained by observations, at 0.25° (about 28 km). There are five views: `"clouds"`, `"moisture"`, `"rain"`, `"temperature"` and `"wind"`. Every map also shows sea-level pressure (lines every 4 hPa) and marks the lowest pressure with an **L**.

In [ ]:
era5 = era5_fields(START, END)

In [ ]:
show_views(era5, "2012-10-29T18")

In [ ]:
animate({"ERA5": era5}, "clouds")

**Look for:** when and where is your event strongest? Which view shows it best?

---
## 3. What does a coarse model see?

NeuralGCM runs on grids of 2.8°, 1.4° or 0.7°. Here ERA5 is averaged onto similar grids.

In [ ]:
show_resolutions(era5, "2012-10-29T18", "wind")

**Look for:** what is lost on the 2.8° grid: the lowest pressure, the strongest wind, the heaviest rain? Is your event still recognizable?

---
## 4. Forecast it with NeuralGCM

The `v1_precip` model is the only one that predicts rain. The others are `"v1/deterministic_2_8_deg.pkl"`, `"v1/deterministic_1_4_deg.pkl"` and `"v1/stochastic_1_4_deg.pkl"`.

In [ ]:
MODEL_NAME = "v1_precip/stochastic_precip_2_8_deg.pkl"
load_model(MODEL_NAME)

In [ ]:
FORECAST_START = "2012-10-24T00"
fc = forecast(FORECAST_START, END)

In [ ]:
animate({"ERA5": era5, "NeuralGCM": fc}, "clouds")

In [ ]:
plot_tracks({"ERA5": era5, "NeuralGCM": fc})
plot_box_summary({"ERA5": era5, "NeuralGCM": fc})

**Look for:** what goes wrong first: the position, the strength or the timing?

---
## 5. Move the starting point

Each forecast takes about a minute, mostly loading the ERA5 starting state.

In [ ]:
STARTS = ["2012-10-20T00", "2012-10-22T00", "2012-10-24T00", "2012-10-26T00"]

runs = {"ERA5": era5}
for start in STARTS:
    runs[pd.Timestamp(start).strftime("start %b %d")] = forecast(start, END)

In [ ]:
plot_tracks(runs)
plot_box_summary(runs)

**Look for:** how many days ahead does the model still see your event? Does an earlier start ever do better?

---
## 6. Optional: a small change far away

Warm the air by 1 K somewhere far from your box, run the same forecast again, and watch where the two forecasts differ.

In [ ]:
NUDGE = (40, 160, 1.0)   # latitude, longitude, kelvin. Also try the other side of the world, e.g. (-40, 60, 1.0)
nudged = forecast(FORECAST_START, END, nudge=NUDGE)
show_spread(nudged, fc, NUDGE)

**Look for:** how soon does the change reach your box, and how big does it get there? What does that mean for forecasts?